# Notebook 00 — Data Acquisition & Alignment
### Memory-Augmented Brain Encoding (rigorous rebuild) · `Mrabbi3/memory-augmented-brain-encoding`

**Why this notebook exists.** The first version of this project trained on *surrogate features derived from the fMRI itself* (PCA/ICA of the BOLD signal), which is circular and leaks the validation set into the inputs. It also compared against a per-TR baseline rather than a model with real temporal context — the issue Stéphane d'Ascoli (TRIBE author) flagged: TRIBE already integrates ~100 TRs via a transformer.

This rebuild fixes the foundation. Here we:
1. Download the **real Algonauts 2025 fMRI** (CC0, 4 subjects, 1000 Schaefer parcels).
2. Download **real pre-extracted stimulus features** (V-JEPA 2 + Whisper + Llama-3.2-3B) — the TRIBE-style trimodal set, already computed by MedARC so we don't process 109 GB of raw video.
3. **Inspect** the exact on-disk structure (never assume a schema).
4. **Align** features to fMRI per episode, with an explicit HRF lag, and verify shapes/NaNs.
5. Save a clean aligned dataset for the modeling notebook (01).

**Prereq (Step 0, already done):** a Colab secret named `HF_TOKEN`, and Llama-3.2 access granted on Hugging Face.

> Run cells top to bottom. When you finish, **copy the printed output of the two cells marked 📋 REPORT BACK** and send them to me — I built the alignment defensively against schemas I haven't directly seen, and those outputs let me lock Notebook 01 to your exact data.

## Config — set your scope here
Defaults are sized to fit a free 15 GB Google Drive (Friends S1–S3, all 4 subjects, 3 backbones ≈ a few GB). Scale up by editing the lists once it runs clean.

In [ ]:
# ============ CONFIG ============
SUBJECTS  = ["sub-01", "sub-02", "sub-03", "sub-05"]   # all 4 Algonauts subjects
TASKS     = ["friends"]               # start with Friends; add "movie10" later
SEASONS   = ["s1", "s2", "s3"]        # Friends seasons to pull features for
BACKBONES = ["vjepa2_avg_feat", "whisper", "Llama-3.2-3B"]  # video / audio / text (TRIBE-style)

HRF_SHIFT_TRS = 3        # features lead fMRI by ~this many TRs (1 TR = 1.49 s). We VALIDATE this in NB 01.
TR_SECONDS    = 1.49

DRIVE_ROOT   = "/content/drive/MyDrive/Research/memory-brain-encoding"
DATA_DIR     = f"{DRIVE_ROOT}/data"
FMRI_DIR     = f"{DATA_DIR}/algonauts_fmri"
FEATURES_DIR = f"{DATA_DIR}/algonauts_features"
ALIGNED_DIR  = f"{DATA_DIR}/aligned"
FMRI_REPO    = "https://github.com/courtois-neuromod/algonauts_2025.competitors.git"
FEAT_REPO    = "medarc/algonauts_2025.features"   # HuggingFace dataset
print("Config loaded.")

## 0. Mount Drive, make folders, authenticate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
for d in [DATA_DIR, FMRI_DIR, FEATURES_DIR, ALIGNED_DIR]:
    os.makedirs(d, exist_ok=True)
print("Drive mounted. Data dir:", DATA_DIR)

In [ ]:
%%capture
# git-annex + datalad are needed to pull the CC0 fMRI dataset
!apt-get -qq install -y git-annex
!pip install -q datalad huggingface_hub h5py matplotlib numpy

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami
login(token=userdata.get('HF_TOKEN'))
print("HF logged in as:", whoami()["name"])

## 1. Download the real fMRI (DataLad, ~2.3 GB for all 4 subjects)
The Algonauts 2025 brain data is shared under a **CC0** license via the Courtois NeuroMod project. We pull only the `fmri/` folder (small) and **skip** `stimuli/` (~109 GB) because the pre-extracted features replace it. We then copy the resolved `.h5` files onto Drive so they survive Colab disconnects.

In [ ]:
import os, glob
os.chdir('/content')
if not os.path.isdir('/content/algonauts_2025.competitors'):
    !datalad install {FMRI_REPO}
os.chdir('/content/algonauts_2025.competitors')

# Pull fMRI for the chosen subjects only
for sub in SUBJECTS:
    print(f"--- datalad get fmri/{sub} ---")
    !datalad get -r -J4 fmri/{sub}

# Copy the actual .h5 content to Drive (follow annex symlinks with -L)
src = glob.glob('/content/algonauts_2025.competitors/fmri/**/*.h5', recursive=True)
print(f"\nResolved {len(src)} fMRI .h5 files; copying to Drive...")
for f in src:
    dest = os.path.join(FMRI_DIR, os.path.basename(f))
    if not os.path.exists(dest):
        !cp -L "{f}" "{dest}"

print("\nfMRI files on Drive:")
for f in sorted(glob.glob(FMRI_DIR + '/*.h5')):
    print(f"  {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.0f} MB)")

## 2. Download pre-extracted features (HuggingFace, a few GB)
These are the real TRIBE-style backbones: **V-JEPA 2** (video), **Whisper** (audio), **Llama-3.2-3B** (text). Files are per-episode, e.g. `vjepa2_avg_feat/friends/s1/friends_s01e01a.h5`.

In [ ]:
from huggingface_hub import snapshot_download

patterns = []
for bk in BACKBONES:
    for task in TASKS:
        if task == "friends":
            for s in SEASONS:
                patterns.append(f"{bk}/friends/{s}/*")
        else:
            patterns.append(f"{bk}/{task}/*")
print("Download patterns:")
for p in patterns: print("  ", p)

snapshot_download(
    repo_id=FEAT_REPO, repo_type="dataset",
    allow_patterns=patterns, local_dir=FEATURES_DIR,
)

import glob, os
feat_files = glob.glob(FEATURES_DIR + "/**/*.h5", recursive=True)
print(f"\nDownloaded {len(feat_files)} feature files.")
for bk in BACKBONES:
    n = len(glob.glob(FEATURES_DIR + f"/{bk}/**/*.h5", recursive=True))
    print(f"  {bk}: {n} episode files")

## 3. 📋 REPORT BACK #1 — Inspect the exact on-disk structure
**Look before you compute.** This prints the real internal layout of both file types so we don't assume a schema. **Send me this cell's output.**

In [ ]:
import h5py, glob, os, numpy as np

def find_fmri(sub, task):
    c = [f for f in glob.glob(FMRI_DIR + "/*.h5") if sub in f and task in os.path.basename(f).lower()]
    return c[0] if c else None

print("="*70, "\nfMRI FILE STRUCTURE\n", "="*70, sep="")
f_fmri = find_fmri("sub-01", "friends")
print("file:", f_fmri)
with h5py.File(f_fmri, "r") as h:
    keys = list(h.keys())
    print(f"# datasets (movie segments): {len(keys)}")
    print("first 10 keys:", keys[:10])
    k0 = keys[0]
    print(f"example dataset '{k0}': shape={h[k0].shape}, dtype={h[k0].dtype}")

print("\n" + "="*70, "\nFEATURE FILE STRUCTURE\n", "="*70, sep="")
for bk in BACKBONES:
    ff = sorted(glob.glob(FEATURES_DIR + f"/{bk}/**/*.h5", recursive=True))
    if not ff:
        print(f"{bk}: (no files)"); continue
    print(f"\n{bk}  ->  {os.path.relpath(ff[0], FEATURES_DIR)}")
    with h5py.File(ff[0], "r") as h:
        for k in h.keys():
            try:
                print(f"   key '{k}': shape={h[k].shape}, dtype={h[k].dtype}")
            except Exception as e:
                print(f"   key '{k}': (group) {list(h[k].keys())[:5]}")

## 4. 📋 REPORT BACK #2 — Align one episode (the make-or-break step)
We match an episode across feature backbones and the fMRI, trim to a common length, apply the HRF lag, z-score, and check for NaNs. **Send me this cell's output too** — the final `X` and `y` shapes confirm the pipeline is sane.

In [ ]:
import re

def episode_id(path):
    m = re.search(r's\d{2}e\d{2}[a-d]?', os.path.basename(path))
    return m.group(0) if m else os.path.basename(path).replace('.h5','')

def load_2d(path):
    with h5py.File(path, "r") as h:
        for k in h.keys():
            a = h[k][:]
            if getattr(a, "ndim", 0) == 2:
                return np.asarray(a, dtype=np.float32)
    return None

def fmri_segment(fmri_path, ep):
    with h5py.File(fmri_path, "r") as h:
        m = [k for k in h.keys() if ep in k]
        if not m: return None, None
        return np.asarray(h[m[0]][:], dtype=np.float32), m[0]

# pick the first episode that exists for the first backbone
ref = sorted(glob.glob(FEATURES_DIR + f"/{BACKBONES[0]}/friends/**/*.h5", recursive=True))[0]
ep = episode_id(ref)
print("Aligning episode:", ep)

feat_arrays, dims = [], []
for bk in BACKBONES:
    cand = [p for p in glob.glob(FEATURES_DIR + f"/{bk}/friends/**/*.h5", recursive=True) if ep in p]
    if not cand:
        print(f"  [warn] no {bk} feature for {ep}"); continue
    a = load_2d(cand[0]); feat_arrays.append(a); dims.append((bk, a.shape))
    print(f"  {bk}: {a.shape}")

y_seg, key = fmri_segment(find_fmri("sub-01", "friends"), ep)
print("  fMRI segment:", (None if y_seg is None else y_seg.shape), " key:", key)

# common length across all feature modalities and fMRI
T = min([a.shape[0] for a in feat_arrays] + [y_seg.shape[0]])
X = np.concatenate([a[:T] for a in feat_arrays], axis=1)
y = y_seg[:T]

# HRF lag: features lead fMRI -> pair y[t] with X[t - shift]
s = HRF_SHIFT_TRS
X, y = X[: T - s], y[s:T]

# z-score each column over time
X = (X - X.mean(0)) / (X.std(0) + 1e-8)
y = (y - y.mean(0)) / (y.std(0) + 1e-8)

print("\n--- RESULT ---")
print("feature dims per backbone:", dims)
print(f"concatenated feature dim: {X.shape[1]}")
print(f"aligned X (features): {X.shape}")
print(f"aligned y (fMRI):     {y.shape}")
print("HRF shift (TRs):", s, f"(~{s*TR_SECONDS:.1f} s)")
print("NaNs -> X:", bool(np.isnan(X).any()), " y:", bool(np.isnan(y).any()))

In [ ]:
# Quick visual sanity check: one feature channel and one high-variance parcel over time
import matplotlib.pyplot as plt
parcel = int(np.argmax(y.var(0)))      # most variable parcel
fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax[0].plot(X[:, 0], lw=0.7); ax[0].set_ylabel("feature ch 0 (z)")
ax[0].set_title(f"Episode {ep}: aligned feature vs fMRI (visual check for temporal structure)")
ax[1].plot(y[:, parcel], lw=0.7, color="crimson"); ax[1].set_ylabel(f"parcel {parcel} (z)")
ax[1].set_xlabel("TR"); plt.tight_layout(); plt.show()

## 5. Build & save the aligned dataset (bridge to Notebook 01)
This loops over every episode we have features for, aligns it to each subject's fMRI, and saves per-subject `.npz` files (`X`, `y`, episode boundaries) into `aligned/`. Notebook 01 just loads these. We demo on `sub-01` so you can confirm it works; flip `RUN_ALL_SUBJECTS = True` to build all four.

In [ ]:
RUN_ALL_SUBJECTS = False   # set True once the demo looks right

def build_subject(sub, task="friends"):
    fmri_path = find_fmri(sub, task)
    if fmri_path is None:
        print(f"  [skip] no fMRI for {sub}/{task}"); return
    # episodes available for ALL chosen backbones
    ep_to_paths = {}
    for bk in BACKBONES:
        for p in glob.glob(FEATURES_DIR + f"/{bk}/{task}/**/*.h5", recursive=True):
            ep_to_paths.setdefault(episode_id(p), {})[bk] = p
    eps = sorted([e for e, d in ep_to_paths.items() if all(bk in d for bk in BACKBONES)])

    Xs, ys, bounds, used = [], [], [0], []
    with h5py.File(fmri_path, "r") as h:
        fkeys = list(h.keys())
        for ep in eps:
            mk = [k for k in fkeys if ep in k]
            if not mk: continue
            feats = [load_2d(ep_to_paths[ep][bk]) for bk in BACKBONES]
            yseg = np.asarray(h[mk[0]][:], dtype=np.float32)
            T = min([f.shape[0] for f in feats] + [yseg.shape[0]])
            Xe = np.concatenate([f[:T] for f in feats], axis=1)
            ye = yseg[:T]
            s = HRF_SHIFT_TRS
            Xe, ye = Xe[:T-s], ye[s:T]
            Xs.append(Xe); ys.append(ye); bounds.append(bounds[-1] + Xe.shape[0]); used.append(ep)
    if not Xs:
        print(f"  [skip] no aligned episodes for {sub}"); return
    X = np.concatenate(Xs, 0); y = np.concatenate(ys, 0)
    # z-score globally per column
    X = (X - X.mean(0)) / (X.std(0) + 1e-8)
    y = (y - y.mean(0)) / (y.std(0) + 1e-8)
    out = os.path.join(ALIGNED_DIR, f"{sub}_{task}_aligned.npz")
    np.savez_compressed(out, X=X.astype(np.float32), y=y.astype(np.float32),
                        bounds=np.array(bounds), episodes=np.array(used),
                        hrf_shift=HRF_SHIFT_TRS, backbones=np.array(BACKBONES))
    print(f"  {sub}: {len(used)} episodes -> X {X.shape}, y {y.shape}  saved {os.path.basename(out)}")

targets = SUBJECTS if RUN_ALL_SUBJECTS else ["sub-01"]
for sub in targets:
    build_subject(sub, "friends")
print("\nDone. Aligned files in:", ALIGNED_DIR)

## Wrap-up
If the shapes printed in cells 3–5 look sane (feature TRs ≈ fMRI TRs per segment, no NaNs, `X`/`y` aligned), the foundation is solid — which is more than the previous version ever had.

**Send me the output of cells 3 and 4.** With your exact key names, feature dimensions, and aligned shapes, I'll build **Notebook 01**: a fair encoder (real temporal context, not per-TR), the **context-length sweep** that maps where longer integration helps across cortex, multi-subject training, and a **noise ceiling** so the R-values are interpretable.